# SAEDtechApp | Kobo XLSForm audit & handover (Google Colab)

**What this notebook does:** checks two XLSForm workbook structures, creates a field catalogue and a safe handover summary. It does **not** download Kobo submissions, invent a weighted score, or connect the dashboard to live records.

**Inputs (upload both):**
- `SAWSO_School_Assessment_Master_v0.5_Handover.xlsx` — eight-criterion School Assessment
- `SAWSO_VF_Combined_Sections_A_B_C_Overview.xlsx` — 14-condition School Evaluation sample

It uses **Python's standard library only** to read XLSX metadata. No passwords, tokens, sensitive school responses or external API access are required. Keep source files private unless SAWSO approves publication.

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from xml.etree import ElementTree as ET
import csv, json, re, zipfile, posixpath

OUT = Path("saedtech_handover")
OUT.mkdir(exist_ok=True)
print("Output directory:", OUT.resolve())

## 1. Upload the two source XLSForms

In Google Colab the following cell opens the upload dialog. Outside Colab, put the two XLSX files in the notebook's working folder.

In [ ]:
try:
    from google.colab import files  # type: ignore
    uploaded = files.upload()
    paths = [Path(name) for name in uploaded if name.lower().endswith(".xlsx")]
except ImportError:
    paths = sorted(Path(".").glob("*.xlsx"))
assert len(paths) >= 2, "Please upload both assessment and evaluation XLSForms."
print("Found:", *[p.name for p in paths], sep="\n - ")

## 2. Read XLSForm tables safely

A small, read-only XLSX parser extracts `survey`, `choices`, and `settings`. This does not modify the form or require a spreadsheet engine.

In [ ]:
NS = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
      "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships"}
R_ID = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"
REL_ID = "Id"
def read_sheet(path, sheet_name):
    with zipfile.ZipFile(path) as z:
        wb = ET.fromstring(z.read("xl/workbook.xml"))
        rels = ET.fromstring(z.read("xl/_rels/workbook.xml.rels"))
        paths_by_id = {e.attrib[REL_ID]: e.attrib["Target"] for e in rels}
        sheet = next((s for s in wb.findall("m:sheets/m:sheet", NS)
                      if s.attrib.get("name") == sheet_name), None)
        if sheet is None: raise ValueError(f"{path.name} missing sheet {sheet_name}")
        target = paths_by_id[sheet.attrib[R_ID]]
        member = target.lstrip("/") if target.startswith("/") else posixpath.normpath("xl/" + target)
        strings = []
        if "xl/sharedStrings.xml" in z.namelist():
            shared = ET.fromstring(z.read("xl/sharedStrings.xml"))
            strings = ["".join((t.text or "") for t in si.iter("{%s}t" % NS["m"]))
                       for si in shared.findall("m:si", NS)]
        tree = ET.fromstring(z.read(member))
        rows = []
        for row in tree.findall("m:sheetData/m:row", NS):
            values = {}
            for c in row.findall("m:c", NS):
                match = re.match(r"([A-Z]+)", c.attrib.get("r", ""))
                if not match: continue
                col = 0
                for letter in match.group(1): col = col * 26 + ord(letter) - 64
                val_node = c.find("m:v", NS)
                raw = val_node.text if val_node is not None else ""
                if c.attrib.get("t") == "s" and raw: raw = strings[int(raw)]
                if c.attrib.get("t") == "inlineStr":
                    raw = "".join((t.text or "") for t in c.iter("{%s}t" % NS["m"]))
                values[col - 1] = raw or ""
            rows.append([values.get(i, "") for i in range(max(values, default=-1) + 1)])
    return rows

def as_dicts(rows):
    if not rows: return []
    heads = [str(v or "").strip() for v in rows[0]]
    return [{key: (row[i] if i < len(row) else "") for i, key in enumerate(heads) if key}
            for row in rows[1:] if any(str(v).strip() for v in row)]

## 3. Audit structural integrity

Checks include duplicate field names, group/repeat nesting, missing choice lists and references to unknown `${fields}`. A clean result is **not** equivalent to successful Kobo import or field validation.

In [ ]:
SELECT_TYPES = {"select_one", "select_multiple"}
BEGIN_TO_END = {"begin_group": "end_group", "begin_repeat": "end_repeat"}
REF = re.compile(r"\$\{([^}]+)\}")

def audit_xlsform(path):
    survey, choices, settings = (as_dicts(read_sheet(path, tab)) for tab in ("survey", "choices", "settings"))
    field_rows = [r for r in survey if r.get("name") and not str(r.get("type","")).startswith(("begin_", "end_"))]
    ids = [r["name"] for r in field_rows]
    known = set(ids)
    lists = {r.get("list_name","") for r in choices if r.get("list_name")}
    problems, stack = [], []
    for i, r in enumerate(survey, start=2):
        kind = str(r.get("type","")).split(" ", 1)[0]
        if kind in BEGIN_TO_END:
            stack.append((kind, i))
        elif kind in BEGIN_TO_END.values():
            needed = "begin_" + kind.removeprefix("end_")
            if not stack or stack[-1][0] != needed:
                problems.append(f"row {i}: unmatched {kind}")
            else:
                stack.pop()
        if kind in SELECT_TYPES:
            type_text = str(r.get("type",""))
            chosen = type_text.split(" ",1)[1] if " " in type_text else ""
            if chosen not in lists:
                problems.append(f"row {i}: missing choice list {chosen!r}")
        for key in ("required", "relevant", "constraint", "calculation"):
            for name in REF.findall(str(r.get(key,""))):
                if name not in known:
                    problems.append(f"row {i} ({key}): unknown field {name}")
    for kind, i in stack:
        problems.append(f"row {i}: missing close for {kind}")
    for name, count in Counter(ids).items():
        if count > 1: problems.append(f"duplicate variable name: {name}")
    question_types = Counter(str(r.get("type","")).split(" ",1)[0] for r in survey
                             if r.get("type") and str(r.get("type")).split(" ",1)[0] not in
                             {"begin_group","end_group","begin_repeat","end_repeat","start","end","today","note","calculate"})
    overview = {
        "file": path.name, "survey_rows": len(survey),
        "question_fields": sum(question_types.values()),
        "groups": sum(str(r.get("type","")) == "begin_group" for r in survey),
        "repeat_groups": sum(str(r.get("type","")) == "begin_repeat" for r in survey),
        "rating_fields": sorted([n for n in ids if re.fullmatch(r"c[1-8]_rating", n)]),
        "section_labels": [r.get(next((k for k in r if k.startswith("label")), "label"), "")
                           for r in survey if str(r.get("type","")) == "begin_group" and
                           (str(r.get("name","")).startswith("criterion_") or str(r.get("name","")).startswith("section_"))],
        "question_type_counts": dict(question_types), "problems": sorted(set(problems))
    }
    return overview, field_rows, choices

audits, all_fields = [], []
for path in paths:
    overview, fields, choices = audit_xlsform(path)
    audits.append(overview)
    for r in fields:
        all_fields.append({
            "source_file": path.name, "field_name": r.get("name",""),
            "type": r.get("type",""),
            "label": next((str(v) for k,v in r.items() if k.startswith("label") and v), ""),
            "required_expression": r.get("required",""),
            "relevance_expression": r.get("relevant","")
        })
    print(path.name, "|", overview["question_fields"], "questions |",
          len(overview["problems"]), "structural issues")
    for p in overview["problems"][:12]: print("  CHECK:", p)

## 4. Export review-ready metadata

Only question metadata and a handover log are exported. No learner data, filled-in school responses, API keys or other private records.

In [ ]:
audit_path = OUT / "xlsform_audit.json"
audit_path.write_text(json.dumps({"generated_at_utc": datetime.now(timezone.utc).isoformat(),
                                    "status": "structural audit only — not Kobo validated",
                                    "forms": audits}, indent=2, ensure_ascii=False), encoding="utf-8")
with (OUT / "form_field_catalog.csv").open("w",newline="",encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["source_file","field_name","type","label",
                                           "required_expression","relevance_expression"])
    writer.writeheader(); writer.writerows(all_fields)
review = """# SAEDtechApp / SAWSO — handover checklist

- [ ] Confirm the approved active-school register and school-to-managing-office mapping
- [ ] Pilot both Kobo forms with school-level staff and review the time burden
- [ ] Obtain approval for assessment cadence, scoring and provisional ratings
- [ ] Obtain approval for the separate Evaluation conditions and country standards
- [ ] Define who can submit, review, correct and approve data at each level
- [ ] Define sensitive-data handling, safeguarding and audit trails
- [ ] Connect approved Kobo exports to the register and reporting model
- [ ] Replace illustrative dashboard figures only after validation
- [ ] Test low-connectivity/offline workflows with local implementers

This notebook did not deploy a live platform or determine school compliance.
"""
(OUT / "HANDOVER_CHECKLIST.md").write_text(review, encoding="utf-8")
print("Outputs:", *[str(p) for p in OUT.iterdir()],sep="\n - ")

## 5. Download a handover ZIP

The ZIP contains the audit report, field catalogue and checklist. The source XLSForms are intentionally excluded to avoid accidental publication or sharing.

In [ ]:
import shutil
archive = shutil.make_archive("SAEDtechApp_Colab_Handover", "zip", root_dir=OUT)
print("Created:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Download this file from the notebook folder:", archive)

## Recommended next step

Review the audit, then test both forms directly in KoboToolbox using fictional cases. Confirm local permissions and workflows before integrating actual school data. The four dashboard views currently remain an **illustrative static prototype**, not a live reporting application.